# Lab 08: Structured Streaming

## Objectives
- Read a stream from JSON files using readStream
- Apply transformations on streaming DataFrames
- Write stream output to Delta tables using writeStream
- Understand output modes: append, update, complete
- Use triggers: availableNow, processingTime

## Exam Domain
- **Structured Streaming — 10%**

## Estimates
- **Time:** ~35 minutes
- **Cost:** $1-2 (serverless)
- **Compute:** Serverless

![Streaming Pipeline](../assets/diagrams/lab-08-streaming-pipeline.png)

In [ ]:
USE_CATALOG = "spark_lab_guide"
USE_SCHEMA = "default"

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

GITHUB_RAW_PATH = "/Volumes/spark_lab_guide/default/github_archive"
CHECKPOINT_BASE = "/Volumes/spark_lab_guide/default/github_archive/_checkpoints"

## A. Batch vs Streaming

In batch processing, you read all data at once. In streaming, data arrives continuously and Spark processes it incrementally. Structured Streaming uses the same DataFrame API — just swap `read` for `readStream` and `write` for `writeStream`.

In [ ]:
# Batch read — processes all files at once
batch_df = spark.read.json(GITHUB_RAW_PATH)
print(f"Batch read: {batch_df.count()} rows")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType, ArrayType

# For streaming, schema MUST be provided — inference is not supported
# Use the schema from our batch read
stream_schema = batch_df.schema

# Streaming read — processes files as they arrive
stream_df = (
    spark.readStream
    .schema(stream_schema)
    .json(GITHUB_RAW_PATH)
)

# This is a streaming DataFrame — note isStreaming=True
print(f"Is streaming: {stream_df.isStreaming}")

> **Exam Tip:** Streaming requires an explicit schema — `inferSchema` is not supported for streaming sources. Always provide a schema when calling `readStream`.

## B. Transformations on Streams

You can apply the same DataFrame transformations to streaming DataFrames: filter, select, groupBy, withColumn, etc. The key constraint: you cannot call actions like `count()` or `show()` on a streaming DataFrame — you must write it to a sink.

In [ ]:
from pyspark.sql.functions import col, to_timestamp, hour

# Transform the stream — same API as batch
transformed_stream = (
    stream_df
    .select(
        col("id"),
        col("type"),
        col("actor.login").alias("username"),
        col("repo.name").alias("repo"),
        to_timestamp(col("created_at")).alias("event_time"),
    )
    .filter(col("type").isNotNull())
)

## C. Writing a Stream — append Mode

`writeStream` sends processed data to a sink. The **output mode** determines what gets written:

| Mode | Writes | Use When |
|------|--------|----------|
| `append` | Only new rows | No aggregations, or aggregations with watermark |
| `update` | Changed rows only | Aggregations (updates existing keys) |
| `complete` | Entire result table | Aggregations (replaces all output) |

In [ ]:
# Write stream in append mode — each new file's rows are appended to the Delta table
query_append = (
    transformed_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/append_stream")
    .trigger(availableNow=True)  # Process all available data, then stop
    .toTable("github_stream_append")
)

# Wait for the stream to finish processing
query_append.awaitTermination()
print("Append stream complete.")
spark.sql("SELECT COUNT(*) AS rows FROM github_stream_append").show()

> **Exam Tip:** Every streaming query MUST have a checkpoint location. Checkpoints track which data has been processed so the stream can resume from where it left off after a failure. Without checkpoints, you get duplicate processing.

## D. Aggregation with complete Mode

For aggregations, use `complete` mode (writes entire result table) or `update` mode (writes only changed rows).

In [ ]:
from pyspark.sql.functions import count

# Aggregation on the stream — count events by type
event_counts_stream = (
    stream_df
    .groupBy("type")
    .agg(count("*").alias("event_count"))
)

# Write with complete mode — the entire aggregation result is written each time
query_complete = (
    event_counts_stream
    .writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/complete_stream")
    .trigger(availableNow=True)
    .toTable("github_stream_complete")
)

query_complete.awaitTermination()
print("Complete mode stream done.")
spark.sql("SELECT * FROM github_stream_complete ORDER BY event_count DESC").show()

## E. Trigger Types

Triggers control when the stream processes data:

| Trigger | Behavior |
|---------|----------|
| `availableNow=True` | Process all available data, then stop. Best for batch-like streaming. |
| `processingTime="10 seconds"` | Process in micro-batches every 10 seconds (continuous). |
| `once=True` | (Deprecated) Process one micro-batch, then stop. Use `availableNow` instead. |
| No trigger | Process as fast as possible (continuous). |

In [ ]:
# Demonstrate processingTime trigger
# NOTE: processingTime is NOT supported on serverless — it requires a cluster
# On a cluster, you would use:
#   .trigger(processingTime="10 seconds")
# This runs continuously in micro-batches until stopped.

# On serverless, we demonstrate with availableNow instead:
query_continuous = (
    transformed_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/continuous_stream")
    .trigger(availableNow=True)
    .toTable("github_stream_continuous")
)

query_continuous.awaitTermination()
print("Stream complete (using availableNow trigger).")
print("On a cluster, processingTime trigger would run continuously.")
spark.sql("SELECT COUNT(*) AS rows FROM github_stream_continuous").show()

## F. Monitoring Active Streams

In [ ]:
# List active streaming queries
active = spark.streams.active
print(f"Active streams: {len(active)}")
for q in active:
    print(f"  Name: {q.name}, ID: {q.id}, Status: {q.status}")

## Exam-Style Review

**Q1.** Can you use `inferSchema=True` with `readStream`?
- A) Yes — it works the same as batch
- B) No — streaming requires an explicit schema
- C) Yes — but only for Parquet files
- D) Yes — but it adds latency

**Q2.** What is the purpose of the checkpoint location in streaming?
- A) To store the final output
- B) To cache intermediate results for faster processing
- C) To track processing progress so the stream can resume after failure
- D) To store the schema of the stream

**Q3.** Which output mode writes the entire aggregation result every time?
- A) append
- B) update
- C) complete
- D) overwrite

**Q4.** What does `trigger(availableNow=True)` do?
- A) Processes one micro-batch and stops
- B) Processes all available data and stops
- C) Runs continuously with no delay
- D) Waits until new data arrives

**Q5.** Can you call `df.count()` on a streaming DataFrame?
- A) Yes — it returns the current count
- B) No — you must write to a sink to see results
- C) Yes — but only after calling `awaitTermination()`
- D) Yes — but it blocks until the stream stops

### Answers
- **Q1: B** — Schema inference is not supported for streaming. You must provide an explicit schema.
- **Q2: C** — Checkpoints track which data has been processed, enabling exactly-once fault-tolerant processing.
- **Q3: C** — `complete` mode writes the entire result table each time. `update` writes only changed rows.
- **Q4: B** — `availableNow` processes all currently available data, then stops. It replaces the deprecated `once` trigger.
- **Q5: B** — Actions like `count()`, `show()`, and `collect()` are not supported on streaming DataFrames. Use `writeStream` to a sink.

## Key Takeaways
- Streaming uses the same DataFrame API: `readStream` instead of `read`, `writeStream` instead of `write`
- Schema must be provided explicitly — no inference for streaming
- Output modes: `append` (new rows), `update` (changed rows), `complete` (full result)
- Checkpoints are required for fault tolerance
- `trigger(availableNow=True)` is the modern way to do batch-like streaming
- Cannot call actions (`count()`, `show()`) on streaming DataFrames

**Next:** [Lab 09 — Delta Lake, Pandas API & Spark Connect](09-delta-lake-pandas-api-spark-connect.ipynb)

In [ ]:
# Stop any active streams
for q in spark.streams.active:
    q.stop()
    print(f"Stopped stream: {q.id}")

# Drop streaming output tables
spark.sql("DROP TABLE IF EXISTS github_stream_append")
spark.sql("DROP TABLE IF EXISTS github_stream_complete")
spark.sql("DROP TABLE IF EXISTS github_stream_continuous")

# Clean up checkpoint directories
dbutils.fs.rm(CHECKPOINT_BASE, True)
print("Lab 08 cleanup complete.")